In [1]:
import pandas as pd
import numpy as np
from math import sin, cos, sqrt, atan2, radians
from global_land_mask import globe
from urllib.parse import urlencode
import math
import datetime
import xlsxwriter
import datetime
from helper import *
today = datetime.date.today()

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
file_name = 'BCT_KINGElvis.xlsx'
detail_df=pd.read_excel("details_project_od_excel_BCT.xlsx",sheet_name='STOPS')
df=pd.read_csv('elvisbroward2023obweekday_export_odbc(db06).csv')
elvis_df=pd.read_excel(file_name,sheet_name='Elvis_Review')

In [4]:
def remove_test_records(df):
    df = df[df['INTERV_INIT']!=999]
    return df

df=remove_test_records(df)
elvis_df=remove_test_records(elvis_df)

In [5]:
def get_removal_marked_records(df: pd.DataFrame) -> pd.DataFrame:
    """
    Get rows from a DataFrame where 'Final_Usage' or 'FINAL_USAGE' is marked as 'Remove' or 'remove'.
    
    Args:
        df (pd.DataFrame): Input DataFrame
    
    Returns:
        pd.DataFrame: DataFrame containing the rows that should be removed
    """
    # Using the 'isin' function to check multiple values at once and casefold for case-insensitive matching
    columns_to_check = ['Final_Usage', 'FINAL_USAGE']
    
    for column in columns_to_check:
        if column in df.columns:
            return df[df[column].apply(lambda x: str(x).casefold()) == 'remove']
    
    return pd.DataFrame()

In [15]:
df_removals = get_removal_marked_records(elvis_df)

In [16]:
df_removals

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,...,DESTIN_TRANSPORTCode,DESTIN_TRANSPORT,_INTRV_NOTE,ELVIS_STATUS,ELVIS_COMMENT,ROUTE_STATUS,Stops_Status,Test_Status,ROUTE_ONLY,Unnamed: 30
51,2023-09-22 00:00:00,5448,No 5 MIN,No 5 MIN,Remove,Test/No 5 MIN,,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,Elvis_Review,Elvis_Review,Tested,NaN,NaN
52,2023-09-22 00:00:00,5451,No 5 MIN,No 5 MIN,Remove,Test/No 5 MIN,,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,Elvis_Review,Elvis_Review,Tested,NaN,NaN
53,2023-09-22 00:00:00,5453,No 5 MIN,No 5 MIN,Remove,Test/No 5 MIN,,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,Elvis_Review,Elvis_Review,Tested,NaN,NaN
56,2023-09-22 00:00:00,5459,No 5 MIN,No 5 MIN,Remove,Test/No 5 MIN,,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,Elvis_Review,Elvis_Review,Tested,NaN,NaN
57,2023-09-22 00:00:00,5460,No 5 MIN,No 5 MIN,Remove,Test/No 5 MIN,,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,Elvis_Review,Elvis_Review,Tested,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10647,2023-10-28 00:00:00,22152,Test/No 5 MIN,NaN,Remove,Test/No 5 MIN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,Review,Review,Tested,NaN,NaN
10651,2023-10-28 00:00:00,22156,Test/No 5 MIN,NaN,Remove,Test/No 5 MIN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,Review,Review,Tested,NaN,NaN
10657,2023-10-28 00:00:00,22169,HereAPI,"Kashish, Brian",Remove,Origin or Destination Greater than 50 Miles,NaN,Elvis_Review,Elvis_Review,NaN,...,9,Be picked up by someone,NaN,NaN,NaN,Review,Review,Tested,NaN,NaN
10662,2023-10-28 00:00:00,22174,HereAPI,"Kashish, Brian",Remove,No Info,NaN,Approved,Elvis_Review,NaN,...,1,Walk,NaN,NaN,NaN,Approved,Review,Tested,NaN,NaN


In [12]:
def haversine_distance(coord1, coord2):
    R = 3958.8  # Radius of the Earth in miles
    try:
        lat1, lon1 = radians(coord1[0]), radians(coord1[1])
    except:
        return 10000000
    try:
        lat2, lon2 = radians(coord2[0]), radians(coord2[1])
    except:
        lat2, lon2 = coord2.split(',')
        lat2, lon2 = radians(float(lat2)), radians(float(lon2))
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    print(distance)
    return distance

In [8]:
def filter_df_by_ids(df, removals_df, column_name='id'):
    """
    Filters a DataFrame based on a list of IDs from another DataFrame.

    Parameters:
        df (DataFrame): The DataFrame to be filtered.
        removals_df (DataFrame): The DataFrame containing IDs to filter out.
        column_name (str): The name of the column containing the IDs.

    Returns:
        DataFrame: A filtered DataFrame containing only the rows with IDs in removals_df.
    """
    removal_ids = removals_df[column_name]
    filtered_df = df[df[column_name].isin(removal_ids)]
    return filtered_df

In [17]:
df_removals = filter_df_by_ids(df, df_removals)
df_removals

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,...,ELVIS_USER_CHANGE_7_C_DATE,ELVIS_USER_CHANGE_7_C_FIELD,ELVIS_USER_CHANGE_7_C_REASON,INVITE_EMAIL,INVITE_SMS,INVITE_CALL,INVITE_TOKEN,INVITE_STATUS,_REVERSE_TRIP,GENERATABLE_TRIPS
51,5448,2023-09-13 06:03:18,78,en,2023-09-13 06:02:45,2023-09-13 06:03:18,174.211.96.216,https://bct-fl.etc-research.com/,-14400,JSW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
52,5451,2023-09-13 06:11:44,78,en,2023-09-13 06:05:58,2023-09-13 06:11:44,174.211.96.216,https://bct-fl.etc-research.com/index.php/surv...,-14400,JSW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53,5453,2023-09-13 06:26:21,78,en,2023-09-13 06:06:56,2023-09-13 06:26:21,107.77.216.168,https://bct-fl.etc-research.com/index.php/surv...,-14400,BWD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56,5459,2023-09-13 06:14:21,78,en,2023-09-13 06:13:50,2023-09-13 06:14:21,174.211.96.216,https://bct-fl.etc-research.com/,-14400,JSW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
57,5460,2023-09-13 06:16:18,78,en,2023-09-13 06:14:24,2023-09-13 06:16:18,174.211.96.216,https://bct-fl.etc-research.com/index.php/surv...,-14400,JSW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10641,22145,2023-10-26 19:49:28,78,en,2023-10-26 19:49:12,2023-10-26 19:49:28,107.72.178.184,https://bct-fl.etc-research.com/index.php/surv...,-14400,TTR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10645,22151,2023-10-26 20:16:06,78,en,2023-10-26 20:14:05,2023-10-26 20:16:06,107.72.178.184,https://bct-fl.etc-research.com/index.php/surv...,-14400,TTR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10646,22152,2023-10-26 20:16:26,78,en,2023-10-26 20:16:11,2023-10-26 20:16:26,107.72.178.184,https://bct-fl.etc-research.com/index.php/surv...,-14400,TTR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10650,22156,2023-10-26 20:24:28,78,en,2023-10-26 20:24:12,2023-10-26 20:24:28,107.72.178.184,https://bct-fl.etc-research.com/index.php/surv...,-14400,TTR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
def get_boarding_points(cord_df):
    here_boarding_points = []  # List to store boarding points
    print("Processing Routes...")
    
    for index, cord in cord_df.iterrows():
        
        # Initialize to None, in case the second pair doesn't exist
        second_pair = None

        # Search for second pair of coordinates if available
        lat_cols = [col for col in cord_df.columns if '_LAT' in col and col not in ['ORIGIN_ADDRESS_LAT', 'DESTIN_ADDRESS_LAT']]
        long_cols = [col.replace('_LAT', '_LONG') for col in lat_cols]
        
        if len(lat_cols) > 1:
            lat = cord.get(lat_cols[1])
            long = cord.get(long_cols[1])
            
            if not pd.isna(lat) and not pd.isna(long):
                if lat not in [',', ''] and long not in [',', '']:
                    second_pair = (lat, long)

        here_boarding_points.append(second_pair)
    
    # Add the boarding points to the DataFrame
    cord_df['here_boarding_point'] = here_boarding_points
    
    return cord_df

In [11]:
def evaluate_removal_conditions(df):
    # Initialize empty lists to store values for each new column
    REVIEW_REVIEWER = []
    REVIEW_USAGE = []
    FLAG_ALL_EQUAL = []
    FLAG_POSS_HD = []
    FLAG_POSS_TRAN = []
    FLAG_WATER_ORIGIN = []  # Flag for origin in water
    FLAG_WATER_DESTIN = []  # Flag for destination in water
    def sanitize_coordinates(lat, lon):
        # Ensure latitude is within [-90, 90]
        lat = max(min(lat, 90), -90)
        # Ensure longitude is within [-180, 180]
        lon = (lon + 180) % 360 - 180
        return lat, lon

    for index, row in df.iterrows():
        HOME = row['HOME_ADDRESS_LAT'], row['HOME_ADDRESS_LONG']
        # Check for NaN values in ORIGIN and DESTIN
        ORIGIN = None if math.isnan(row['ORIGIN_ADDRESS_LAT']) or math.isnan(row['ORIGIN_ADDRESS_LONG']) else (row['ORIGIN_ADDRESS_LAT'], row['ORIGIN_ADDRESS_LONG'])
        DESTIN = None if math.isnan(row['DESTIN_ADDRESS_LAT']) or math.isnan(row['DESTIN_ADDRESS_LONG']) else (row['DESTIN_ADDRESS_LAT'], row['DESTIN_ADDRESS_LONG'])
        
        BOARDING = row['here_boarding_point']
        
        # Set common REVIEW_REVIEWER
        REVIEW_REVIEWER.append("Recovery Script")

        # Check if ORIGIN or DESTIN are in a body of water using global_land_mask, if they are not None
        is_origin_water = not globe.is_land(*ORIGIN) if ORIGIN is not None else False
        is_destin_water = not globe.is_land(*DESTIN) if DESTIN is not None else False
        FLAG_WATER_ORIGIN.append(is_origin_water)
        FLAG_WATER_DESTIN.append(is_destin_water)
       

        # Three matching points
        if HOME == ORIGIN and HOME == DESTIN and ORIGIN == DESTIN:
            REVIEW_USAGE.append('Remove')
            FLAG_ALL_EQUAL.append(True)
            FLAG_POSS_HD.append(False)
            FLAG_POSS_TRAN.append(False)
        elif is_origin_water or is_destin_water:
            REVIEW_USAGE.append('REVIEW')
            FLAG_ALL_EQUAL.append(False)
            FLAG_POSS_HD.append(False)
            FLAG_POSS_TRAN.append(False)
        # Home possible origin or destination
        elif ORIGIN == DESTIN and (HOME != ORIGIN or HOME != DESTIN) and (HOME != None) and (DESTIN != None):
            REVIEW_USAGE.append('REVIEW')  # 'Use' changed to 'REVIEW'
            FLAG_ALL_EQUAL.append(False)
            FLAG_POSS_HD.append(True)
            FLAG_POSS_TRAN.append(False)
        
        # Possible transfer case
        else:
            # 'Use' changed to 'REVIEW'
            FLAG_ALL_EQUAL.append(False)
            FLAG_POSS_HD.append(False)
            
            if BOARDING is not None and haversine_distance(ORIGIN, BOARDING) > 2:
                FLAG_POSS_TRAN.append(True)
                REVIEW_USAGE.append('REVIEW')
            else:
                FLAG_POSS_TRAN.append(False)
                REVIEW_USAGE.append('Remove')
             

    # Add new columns to the DataFrame
    df['REVIEW_REVIEWER'] = REVIEW_REVIEWER
    df['REVIEW_USAGE'] = REVIEW_USAGE
    df['FLAG_ALL_EQUAL'] = FLAG_ALL_EQUAL
    df['FLAG_POSS_HD'] = FLAG_POSS_HD
    df['FLAG_POSS_TRAN'] = FLAG_POSS_TRAN
    df['FLAG_WATER_ORIGIN'] = FLAG_WATER_ORIGIN  # Adding new flag column for ORIGIN
    df['FLAG_WATER_DESTIN'] = FLAG_WATER_DESTIN  # Adding new flag column for DESTIN
    flag_columns = ['FLAG_ALL_EQUAL', 'FLAG_POSS_HD', 'FLAG_POSS_TRAN', 'FLAG_WATER_ORIGIN', 'FLAG_WATER_DESTIN']

    # Convert each flag column to integer type
    for col in flag_columns:
        df[col] = df[col].astype(int)
    # Select only the columns you want to keep in the final DataFrame
    return df[['REVIEW_REVIEWER', 'REVIEW_USAGE', 'Date_started', 'id', 'FLAG_ALL_EQUAL', 'FLAG_POSS_HD', 'FLAG_POSS_TRAN', 'FLAG_WATER_ORIGIN', 'FLAG_WATER_DESTIN']]


In [13]:
def make_survey_recovery_sheet(excel_path, df):
    # Generate a unique name for the new workbook using a timestamp
    #current_time = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
    
    excel_path = f"{excel_path}.xlsx"
    
    # Create a Pandas Excel writer using XlsxWriter as the engine.
    with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
        # Write the DataFrame to the Excel workbook
        df.to_excel(writer, sheet_name='_(F0) SURVEY RECOVERY')

In [18]:
bp_df = get_boarding_points(df_removals)
recovery_df = evaluate_removal_conditions(bp_df)
make_survey_recovery_sheet(f'BCT_survey_recovery_{today}', recovery_df)

Processing Routes...
